<a href="https://colab.research.google.com/github/joekobbydarko-del/universal-data-intelligence-demo/blob/feature%2Fdatabase-integration/backend/database/universal_data_intelligence_database_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sqlalchemy psycopg[binary]

In [18]:
from google.colab import userdata

NEON_URL = userdata.get("NEON_DATABASE_URL")
print("Connection string loaded securely.")

Connection string loaded securely.


In [19]:
from sqlalchemy import create_engine, text

engine = create_engine(NEON_URL)

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.scalar())

print("Database connection successful")

1
Database connection successful


In [20]:
from sqlalchemy.orm import declarative_base
from sqlalchemy import Column, Integer, String, Text, DateTime, ForeignKey
from sqlalchemy.sql import func

Base = declarative_base()


class Document(Base):
    __tablename__ = "documents"

    id = Column(Integer, primary_key=True)
    document_name = Column(String(255), nullable=False)
    document_type = Column(String(100))
    file_path = Column(Text)
    upload_date = Column(DateTime, server_default=func.now())
    page_count = Column(Integer)
    processing_status = Column(String(50))


class Record(Base):
    __tablename__ = "records"

    id = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey("documents.id", ondelete="CASCADE"), nullable=False)
    record_type = Column(String(100))
    field_name = Column(String(150), nullable=False)
    field_value = Column(Text)
    page_number = Column(Integer)


class AnalysisResult(Base):
    __tablename__ = "analysis_results"

    id = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey("documents.id", ondelete="CASCADE"), nullable=False)
    analysis_type = Column(String(100), nullable=False)
    result = Column(Text, nullable=False)
    created_at = Column(DateTime, server_default=func.now())

In [21]:
from sqlalchemy import inspect

inspector = inspect(engine)

print("Tables in Neon:")
print(inspector.get_table_names())

for table_name in ["documents", "records", "analysis_results"]:
    print(f"\nColumns in {table_name}:")
    for column in inspector.get_columns(table_name):
        print(column["name"], "-", column["type"])

Tables in Neon:
['documents', 'records', 'analysis_results']

Columns in documents:
id - INTEGER
document_name - VARCHAR(255)
document_type - VARCHAR(100)
file_path - TEXT
upload_date - TIMESTAMP
page_count - INTEGER
processing_status - VARCHAR(50)

Columns in records:
id - INTEGER
document_id - INTEGER
record_type - VARCHAR(100)
field_name - VARCHAR(150)
field_value - TEXT
page_number - INTEGER

Columns in analysis_results:
id - INTEGER
document_id - INTEGER
analysis_type - VARCHAR(100)
result - TEXT
created_at - TIMESTAMP


In [22]:
from sqlalchemy.orm import sessionmaker

SessionLocal = sessionmaker(bind=engine)

with SessionLocal() as session:
    test_document = Document(
        document_name="database_integration_test.pdf",
        document_type="test",
        file_path="test/database_integration_test.pdf",
        page_count=1,
        processing_status="pending"
    )

    session.add(test_document)
    session.commit()
    session.refresh(test_document)

    print("Test document created.")
    print("Document ID:", test_document.id)
    print("Status:", test_document.processing_status)

Test document created.
Document ID: 1
Status: pending


In [23]:
with SessionLocal() as session:
    test_document = (
        session.query(Document)
        .filter(Document.document_name == "database_integration_test.pdf")
        .first()
    )

    test_document.processing_status = "processed"
    session.commit()
    session.refresh(test_document)

    print("Document updated.")
    print("New status:", test_document.processing_status)

Document updated.
New status: processed


In [24]:
with SessionLocal() as session:
    test_document = (
        session.query(Document)
        .filter(Document.document_name == "database_integration_test.pdf")
        .first()
    )

    if test_document:
        session.delete(test_document)
        session.commit()
        print("Test document deleted.")
    else:
        print("Test document not found.")

Test document deleted.
